# Video as space-time patches
**LLMs, Animated · VD.1 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/vd-1-spacetime-patches.ipynb)

A video is a tensor of shape $T \times H \times W \times 3$. Transformers read it as a sequence of **tubelets** (space-time patches): boxes that span $t$ frames and a $p \times p$ pixel square, each flattened into one token. In this notebook you will:

1. build a tiny synthetic video of a moving square,
2. cut it into tubelets and put it back together **exactly**,
3. count tokens for real settings (raw-pixel tubelets and VAE latents),
4. compare the cost of **full** space-time attention with **divided** attention.

Only NumPy and Matplotlib; everything runs on a CPU in a few seconds. About 20 minutes. Solutions are at the end.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)
BG = np.array([20, 24, 40], dtype=np.uint8)       # dark blue background
SQUARE = np.array([232, 119, 46], dtype=np.uint8)  # orange square

## 1. A synthetic video
Write `make_video`. Every frame is the background colour, with a `size × size` orange square whose left edge moves `speed` pixels to the right each frame.

In [ ]:
def make_video(T=16, H=64, W=96, size=12, speed=4):
    """A (T, H, W, 3) uint8 video: an orange square sliding right over a dark background.
    In frame i the square's top-left corner is at row (H - size) // 2, column 4 + speed * i."""
    video = np.zeros((T, H, W, 3), dtype=np.uint8)
    # TODO 1: fill every pixel of every frame with the background colour BG
    # TODO 2: for each frame i, paint a size × size square of colour SQUARE at the position above
    raise NotImplementedError("Fill in make_video")

In [ ]:
v = make_video()
assert v.shape == (16, 64, 96, 3) and v.dtype == np.uint8, (v.shape, v.dtype)
assert (v[0, 0, 0] == BG).all() and (v[0, 26, 4] == SQUARE).all() and (v[0, 26, 16] == BG).all()
assert (v[5, 30, 24] == SQUARE).all() and (v[5, 30, 23] == BG).all()   # frame 5: left edge at 4 + 4*5 = 24
assert (v == SQUARE).all(axis=-1).sum() == 16 * 12 * 12
print("✅ make_video works. Raw size:", v.nbytes, "bytes")

fig, axes = plt.subplots(1, 4, figsize=(10, 2))
for ax, i in zip(axes, [0, 5, 10, 15]):
    ax.imshow(v[i]); ax.set_title(f"frame {i}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Tubelets: video → tokens
Write `patchify(video, t, p)`. The trick is one `reshape`, one `transpose`, one `reshape`:

- split each axis into "which block" × "position inside the block": `(T//t, t, H//p, p, W//p, p, C)`;
- move the three "which block" axes to the front: `(T//t, H//p, W//p, t, p, p, C)`;
- flatten: one row per tubelet.

The row order is (time slab, patch row, patch column), so token $k$ belongs to slab $k \,//\, (\text{rows} \cdot \text{cols})$.

In [ ]:
def patchify(video, t, p):
    """Cut a (T, H, W, C) video into t × p × p tubelets.
    Returns (N, t*p*p*C): one row per tubelet, ordered by (time slab, patch row, patch column);
    each row is the tubelet flattened in (frame, y, x, channel) order."""
    T, H, W, C = video.shape
    assert T % t == 0 and H % p == 0 and W % p == 0, "sizes must divide evenly"
    # TODO 1: reshape to (T//t, t, H//p, p, W//p, p, C)
    # TODO 2: transpose so the three "which tubelet" axes come first: (slab, row, col, t, p, p, C)
    # TODO 3: reshape to (N, t*p*p*C)
    raise NotImplementedError("Fill in patchify")

In [ ]:
tok = patchify(v, t=2, p=16)
assert tok.shape == (8 * 4 * 6, 2 * 16 * 16 * 3), tok.shape          # 192 tokens of 1,536 numbers
assert np.array_equal(tok[0], v[0:2, 0:16, 0:16].reshape(-1))
k = 1 * (4 * 6) + 2 * 6 + 3                                           # slab 1, patch row 2, patch column 3
assert np.array_equal(tok[k], v[2:4, 32:48, 48:64].reshape(-1))
print("✅ patchify works:", tok.shape[0], "tokens, each", tok.shape[1], "numbers")

## 3. And back again
Write `unpatchify`, the exact inverse. No information is lost by patchifying: it's only a re-arrangement. (The learned linear projection that turns each 1,536-number row into a $d$-dimensional token is a separate step.)

In [ ]:
def unpatchify(tokens, shape, t, p):
    """Exact inverse of patchify: (N, t*p*p*C) tokens back to a (T, H, W, C) video."""
    T, H, W, C = shape
    # TODO: undo the three steps of patchify in reverse order (reshape, inverse transpose, reshape)
    raise NotImplementedError("Fill in unpatchify")

In [ ]:
for t, p in [(1, 16), (2, 16), (4, 8), (16, 32)]:
    back = unpatchify(patchify(v, t, p), v.shape, t, p)
    assert back.shape == v.shape and np.array_equal(back, v), (t, p)
print("✅ unpatchify inverts patchify exactly for every tubelet size")

# Look at the tubelet that contains the moving square in slab 1 (frames 2-3)
tok = patchify(v, 2, 16)
has_square = [(tok[i].reshape(2, 16, 16, 3) == SQUARE).all(-1).any() for i in range(len(tok))]
print("tubelets that contain part of the square:", int(np.sum(has_square)), "of", len(tok))

## 4. Counting tokens for real videos
Write `count_tokens(frames, H, W, ft, s, causal=False)`: one token covers `ft` frames and an `s × s` pixel square. For a video VAE plus patches, multiply the factors: a 4 × 8 × 8 VAE with 1 × 2 × 2 patches gives `ft = 4`, `s = 16`. Round up when sizes don't divide evenly.

A **causal** video VAE encodes the first frame on its own, so there are $1 + \lceil (T-1)/f_t \rceil$ time steps.

In [ ]:
def count_tokens(frames, H, W, ft, s, causal=False):
    """Tokens for a video when one token covers `ft` frames and an s × s pixel square.
    causal=True: the first frame is encoded on its own (1 + ceil((frames - 1) / ft) time steps).
    Returns (n_tokens, time_steps, places_per_step)."""
    # TODO 1: time steps (round up; handle causal)
    # TODO 2: places per step = ceil(H / s) * ceil(W / s)
    # TODO 3: return steps * places, steps, places
    raise NotImplementedError("Fill in count_tokens")

In [ ]:
assert count_tokens(240, 720, 1280, 2, 16) == (432_000, 120, 3_600)              # 10 s, 24 fps, ViViT-style 2×16×16
assert count_tokens(241, 720, 1280, 4, 16, causal=True) == (219_600, 61, 3_600)   # 4×8×8 VAE + 2×2 patches
assert count_tokens(81, 480, 832, 4, 16, causal=True) == (32_760, 21, 1_560)      # the lesson's faded example
assert count_tokens(256, 768, 768, 8, 16) == (73_728, 32, 2_304)                  # Movie Gen-style: 8× per axis + 2×2
assert count_tokens(30, 1080, 1920, 1, 16)[2] == 68 * 120                         # 1080/16 = 67.5 rounds up
print("✅ count_tokens matches the lesson's numbers")

## 5. Full vs divided attention
Write `attention_pairs(steps, places)`, returning the number of query-key pairs for **full** space-time attention ($N^2$) and for **divided** attention ($N(S + T')$, where $S$ = places per step and $T'$ = time steps).

In [ ]:
def attention_pairs(steps, places):
    """Query-key pairs per head, per layer, for N = steps * places tokens.
    full:    every token attends to every token           -> N**2
    divided: same place over time + same time step (space) -> N * (steps + places)"""
    # TODO: return (full, divided)
    raise NotImplementedError("Fill in attention_pairs")

In [ ]:
full, div = attention_pairs(61, 3_600)
assert full == 219_600 ** 2 and div == 219_600 * 3_661
assert round(full / div) == 60
assert attention_pairs(4, 12) == (2_304, 768)        # the animation: 48 tokens
print(f"✅ 10 s of 720p: full {full:.2e} pairs, divided {div:.2e} pairs, {full / div:.0f}× fewer")

## Experiment
How do the costs grow with duration? The cell below sweeps 1 to 60 seconds of 720p at 24 fps with a causal 4 × 8 × 8 VAE and 2 × 2 patches. Then try your own changes:

- switch to 1080p or 4K (`H, W`), or to `ft = 8` (8× temporal compression);
- compare the tokens with a 32K-token text prompt and a 1M-token context;
- in the second cell, see how many tubelets actually **change** from one time slab to the next in our toy video. Most don't, which is the redundancy codecs and token-merging methods exploit.

In [ ]:
H, W, fps, ft, s = 720, 1280, 24, 4, 16
secs = np.arange(1, 61)
rows = [count_tokens(int(x * fps) + 1, H, W, ft, s, causal=True) for x in secs]
n = np.array([r[0] for r in rows], dtype=float)
pairs = np.array([attention_pairs(r[1], r[2]) for r in rows], dtype=float)

plt.figure(figsize=(7, 4))
plt.semilogy(secs, n, label="tokens N")
plt.semilogy(secs, pairs[:, 1], label="divided attention pairs")
plt.semilogy(secs, pairs[:, 0], label="full attention pairs")
plt.axhline(32_768 ** 2, color="gray", ls="--", lw=1, label="full attention, 32K-token text")
plt.xlabel("duration (s), 720p at 24 fps"); plt.ylabel("count (log scale)")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

for x in [5, 10, 30, 60]:
    N, st, pl = count_tokens(x * fps + 1, H, W, ft, s, causal=True)
    f, d = attention_pairs(st, pl)
    print(f"{x:>3} s: {N:>9,} tokens | full {f:.1e} | divided {d:.1e} | full / 32K-text = {f / 32_768**2:,.0f}×")

In [ ]:
t, p = 2, 16
tok = patchify(v, t, p).reshape(v.shape[0] // t, -1, t * p * p * 3)   # (slabs, places, numbers)
changed = [int((tok[i] != tok[i - 1]).any(axis=1).sum()) for i in range(1, len(tok))]
print("places per slab:", tok.shape[1])
print("tubelets that differ from the previous slab:", changed)

**Think about it** (answers below the solutions):
1. Doubling the frame rate and halving the duration keeps the number of frames the same. Does it keep the token count the same?
2. Why does a VAE with 4 × 8 × 8 compression plus 2 × 2 patches give *the same* token count as raw $4 \times 16 \times 16$ tubelets, yet work much better for generation?
3. Divided attention needs two layers for information to travel from (time 1, top-left) to (time 9, bottom-right). Why might that matter more for generation than for classification?

---
## Solutions
Only run these if you're stuck. They redefine the functions; re-run the check cells afterwards.

In [ ]:
def make_video(T=16, H=64, W=96, size=12, speed=4):
    """A (T, H, W, 3) uint8 video: an orange square sliding right over a dark background.
    In frame i the square's top-left corner is at row (H - size) // 2, column 4 + speed * i."""
    video = np.zeros((T, H, W, 3), dtype=np.uint8)
    video[...] = BG
    top = (H - size) // 2
    for i in range(T):
        left = 4 + speed * i
        video[i, top:top + size, left:left + size] = SQUARE
    return video

In [ ]:
def patchify(video, t, p):
    """Cut a (T, H, W, C) video into t × p × p tubelets.
    Returns (N, t*p*p*C): one row per tubelet, ordered by (time slab, patch row, patch column);
    each row is the tubelet flattened in (frame, y, x, channel) order."""
    T, H, W, C = video.shape
    assert T % t == 0 and H % p == 0 and W % p == 0, "sizes must divide evenly"
    x = video.reshape(T // t, t, H // p, p, W // p, p, C)
    x = x.transpose(0, 2, 4, 1, 3, 5, 6)          # (slab, row, col, t, p, p, C)
    return x.reshape(-1, t * p * p * C)

In [ ]:
def unpatchify(tokens, shape, t, p):
    """Exact inverse of patchify: (N, t*p*p*C) tokens back to a (T, H, W, C) video."""
    T, H, W, C = shape
    x = tokens.reshape(T // t, H // p, W // p, t, p, p, C)
    x = x.transpose(0, 3, 1, 4, 2, 5, 6)          # (slab, t, row, p, col, p, C)
    return x.reshape(T, H, W, C)

In [ ]:
def count_tokens(frames, H, W, ft, s, causal=False):
    """Tokens for a video when one token covers `ft` frames and an s × s pixel square.
    causal=True: the first frame is encoded on its own (1 + ceil((frames - 1) / ft) time steps).
    Returns (n_tokens, time_steps, places_per_step)."""
    steps = 1 + math.ceil((frames - 1) / ft) if causal else math.ceil(frames / ft)
    places = math.ceil(H / s) * math.ceil(W / s)
    return steps * places, steps, places

In [ ]:
def attention_pairs(steps, places):
    """Query-key pairs per head, per layer, for N = steps * places tokens.
    full:    every token attends to every token           -> N**2
    divided: same place over time + same time step (space) -> N * (steps + places)"""
    n = steps * places
    return n * n, n * (steps + places)

**Think-about-it answers**
1. Yes, as long as the resolution and tokenizer stay the same: tokens depend only on frames × pixels. But the content differs: at a high frame rate consecutive frames are more similar, so the model "spends" tokens on smaller changes.
2. Both give one token per 4 frames and 16 × 16 pixels. With raw tubelets each token is 3,072 raw pixel values; with a VAE each token is $2 \cdot 2 \cdot 16 = 64$ compact latent numbers that already summarize texture and motion, so the generator models a smoother, lower-dimensional space and spends no capacity on imperceptible pixel detail.
3. A generator has to keep *every* pixel consistent with every other at every denoising step (an object moving across the frame must look the same everywhere it goes), so cheap direct links between any two space-time positions help. A classifier only needs a global summary, which divided attention gathers fine.

**Next:** VD.2, video diffusion transformers: how these tokens get generated from noise.